In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import json
import sys

import numpy as np
# Resolve project/backend roots from notebook location.
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd.parent, _cwd.parent.parent, _cwd.parent.parent.parent]
project_root = None
for p in _candidates:
    if (p / "backend").exists():
        project_root = p
        break
if project_root is None:
    raise RuntimeError("Could not resolve project root containing backend/.")

backend_root = project_root / "backend"
if str(backend_root) not in sys.path:
    sys.path.insert(0, str(backend_root))

from autonomous_control.config.randomness import RandomnessConfig, apply_global_seed, derive_seed
from autonomous_control.controller_agent import MPOAgent
from autonomous_control.mpo_config import MPOConfig
from autonomous_control.training_runtime import make_attitude_control_env, run_episode
from environment_definition.constants import RenderMode
from environment_definition.mission_profiles.s01_multiple_targets_fwd_fish import sample_satellite_altitude
from render.render_main import render_from_series
from utils.ml_training.ml_training_utils import create_run_dir, init_run_markdown, append_run_markdown_event

SEED = 7
VIDEOS_PER_CELL = 1
RNG_CFG = RandomnessConfig(seed=SEED)
apply_global_seed(RNG_CFG)
SATELLITE_ALTITUDE = sample_satellite_altitude(seed=derive_seed(SEED, "mission_altitude"))

RUN_ID = f"nb-mpo-{datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S')}"
RUN_DIR = create_run_dir(run_id=RUN_ID)

init_run_markdown(
    RUN_DIR,
    title="Notebook MPO Workflow",
    metadata={
        "seed": SEED,
        "sampled_altitude_km": float(SATELLITE_ALTITUDE.to("km").magnitude),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    },
)

print(f"RUN_DIR: {RUN_DIR}")
print(f"Sampled altitude: {SATELLITE_ALTITUDE}")

warmup_episode_count = 10
train_episode_count = 3
test_episode_count = 1

print(f"""
##########################################
##########################################
Warmup episodes: {warmup_episode_count}
#################################
Training ep:     {train_episode_count}
#################################
Test ep:         {test_episode_count}
##########################################
##########################################
""")


these new constraints need to be wired in somehow:

# New Constraints

## sat model
### control / safety
- max 30 deg sweep in all directions -> encode as RW safety mechanism:
- create safe mode to comply with safety restrictions
#### safety torque cmd constraints
- - if in (30,35) interval reduce torque gradually
#### safe mode activation
- -  if over 35 deg abort and go into "safe mode" -> returning to nadir
- - if over 30 deg and safe mode activation at 35 would let the sat exceed 45 deg due to momentum
- - stay in nadir for 10 s (no external torque control mode allowed)

### 2nd vision sensor
- make an educated guess of what fish eye placement would be useful -> add to 2nd camera init
- educated guess of what FOV for the fish eye would be useful -> add to 2nd camera init

## environment:
- clouds cover a larger area
- clouds move
- vary in height

## mission:
- max 10 images per orbit (memory, downlink constraint)
- capture as many high value observations over a cloud covered area as possible -> benchmark
- images with clouds get less / no reward as not useful
- images need to be captured with fwd motion compensation, get more reward if not blurry

- - implement image capturing mode (when control decides to do the image capturing on a target )

## RL / reward mechanism / KPI:
- calulate a benchmark using a "dummy strategy" for the env initiation

## control agent
- give 2nd camera array
- add action space: "capture image" -> records the "camera ground speed" over fixed time interval starting from action input ending after 2s,

